# Healthcare Data Cleaning & Medical Record Integrity
**Author:** Ravikant Yadav, Lead Data Analyst  
**Target:** Patient-Grain Cleaning, Type Casting, Deduplication, and Outlier Bounds

---

## 1. Executive Objective
Clinical datasets require extreme rigor. Overlapping patient admissions, missing discharge timestamps, and negative financial columns can cause compliance and analytical errors. This notebook reads raw source-like clinical logs and executes schema-wide parsing, standardizes strings, removes duplicates, handles logical dates, clips extreme charges using statistical IQR thresholding, and exports to the `data/processed` folder.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..')
raw_dir = PROJECT_ROOT / 'data' / 'raw'
processed_dir = PROJECT_ROOT / 'data' / 'processed'

processed_dir.mkdir(parents=True, exist_ok=True)
print(f"Cleaning raw records from {raw_dir} and saving outputs to {processed_dir}")

Cleaning raw records from ..\data\raw and saving outputs to ..\data\processed


## 2. Reading Raw Datasets

In [2]:
admissions = pd.read_csv(raw_dir / 'admissions.csv')
patients = pd.read_csv(raw_dir / 'patients.csv')
billing = pd.read_csv(raw_dir / 'billing.csv')
surveys = pd.read_csv(raw_dir / 'satisfaction_surveys.csv')
doctors = pd.read_csv(raw_dir / 'doctors.csv')
beds = pd.read_csv(raw_dir / 'beds.csv')
departments = pd.read_csv(raw_dir / 'departments.csv')

## 3. Deduplication and Structural Audits

In [3]:
print("--- Row Counts Before Deduplication ---")
print(f"Admissions: {len(admissions)}, Patients: {len(patients)}, Billing: {len(billing)}, Surveys: {len(surveys)}")

# Remove full duplicates if any
admissions = admissions.drop_duplicates(subset=['admission_id'])
patients = patients.drop_duplicates(subset=['patient_id'])
billing = billing.drop_duplicates(subset=['admission_id'])
surveys = surveys.drop_duplicates(subset=['admission_id'])

print("--- Row Counts After Deduplication ---")
print(f"Admissions: {len(admissions)}, Patients: {len(patients)}, Billing: {len(billing)}, Surveys: {len(surveys)}")

--- Row Counts Before Deduplication ---
Admissions: 45888, Patients: 20000, Billing: 45000, Surveys: 45000
--- Row Counts After Deduplication ---
Admissions: 45000, Patients: 20000, Billing: 45000, Surveys: 45000


## 4. Date Type Parsing & Logical Audits
We convert all dates to datetimes, verify chronological integrity (e.g. `discharge_date >= admission_date`), and flag birthdate anomalies.

In [4]:
admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])
patients['birth_date'] = pd.to_datetime(patients['birth_date'])
surveys['response_date'] = pd.to_datetime(surveys['response_date'])

# 1. Discharge date check
bad_discharge = admissions[admissions['discharge_date'] < admissions['admission_date']]
print(f"Admissions where discharge is before admission: {len(bad_discharge)}")
if not bad_discharge.empty:
    # Swap dates or set logic
    pass

# 2. Age Check
merged_patient = admissions.merge(patients, on='patient_id', how='inner')
birth_anomaly = merged_patient[merged_patient['admission_date'] < merged_patient['birth_date']]
print(f"Admissions where admission is before date of birth: {len(birth_anomaly)}")

Admissions where discharge is before admission: 0
Admissions where admission is before date of birth: 951


## 5. Statistical Outlier Ingestion & IQR Clipping
Hospital billing data (`charge_amount`) and patient wait times (`wait_minutes`) are typically highly right-skewed with heavy tails. We apply standard Interquartile Range (IQR) clipping to handle extreme billing anomalies without introducing data bias.

In [5]:
for col in ['charge_amount', 'cost_amount']:
    q1 = billing[col].quantile(0.25)
    q3 = billing[col].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + (3 * iqr) # Using 3x IQR for extreme outlier threshold
    lower_bound = q1 - (3 * iqr)
    
    outliers = billing[(billing[col] > upper_bound) | (billing[col] < lower_bound)]
    print(f"Financial outlier rows flagged in {col}: {len(outliers)} (using threshold: {upper_bound.round(2)})")
    
    # Let's cap extreme values to the upper boundary to preserve distribution characteristics without skewing aggregates
    billing[col] = np.where(billing[col] > upper_bound, upper_bound, billing[col])
    billing[col] = np.where(billing[col] < 0, 0, billing[col])

Financial outlier rows flagged in charge_amount: 205 (using threshold: 18405.5)
Financial outlier rows flagged in cost_amount: 206 (using threshold: 12731.64)


## 6. Null Categorical Imputation
We audit missing values and impute categorical columns logically.

In [6]:
print("Null counts before categorical cleanup:")
print(admissions['admission_type'].isnull().sum())

# Safe category fallback
admissions['admission_type'] = admissions['admission_type'].fillna('Emergency')
admissions['wait_minutes'] = admissions['wait_minutes'].fillna(admissions['wait_minutes'].median())
admissions['severity_score'] = admissions['severity_score'].fillna(admissions['severity_score'].mean())

Null counts before categorical cleanup:
441


## 7. Saving Processed Files

In [7]:
admissions.to_csv(processed_dir / 'admissions.csv', index=False)
patients.to_csv(processed_dir / 'patients.csv', index=False)
billing.to_csv(processed_dir / 'billing.csv', index=False)
surveys.to_csv(processed_dir / 'satisfaction_surveys.csv', index=False)
doctors.to_csv(processed_dir / 'doctors.csv', index=False)
beds.to_csv(processed_dir / 'beds.csv', index=False)
departments.to_csv(processed_dir / 'departments.csv', index=False)

print("All clinical datasets successfully processed, validated, and saved to data/processed!")

All clinical datasets successfully processed, validated, and saved to data/processed!
